<a href="https://colab.research.google.com/github/Nancy-Shi/Individual_Infection_Network/blob/main/0311_1000_SF_SW_clean_pair_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Compare scale-free and small-world node pairs experiment**

In [2]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
import networkx as nx
import matplotlib.pyplot as plt
import pickle
import random
import numpy as np
import pandas as pd
import os
import sympy as sp
from sklearn.metrics import r2_score, mean_squared_error
from scipy.optimize import curve_fit
import math

In [7]:
import os
import pickle
import random
import numpy as np
import pandas as pd
import networkx as nx

def repeated_pairwise_centrality_validation(
    graph_path,
    attack_path,
    label,
    outdir,
    n_random_pairs=100,
    n_repeats=100,
    seed=42
):

    with open(graph_path, "rb") as f:
        G = pickle.load(f)

    df_attack = pd.read_csv(attack_path).copy()

    if "Node" not in df_attack.columns or "AverageAttackRate" not in df_attack.columns:
        raise ValueError(f"{attack_path} must contain columns: Node, AverageAttackRate")

    degree_centrality = nx.degree_centrality(G)
    betweenness_centrality = nx.betweenness_centrality(G)

    df_cent = pd.DataFrame({
        "Node": list(G.nodes()),
        "degree_centrality": [degree_centrality[n] for n in G.nodes()],
        "betweenness_centrality": [betweenness_centrality[n] for n in G.nodes()]
    })

    df = df_attack.merge(df_cent, on="Node", how="inner").copy()
    nodes = df["Node"].tolist()

    if len(nodes) < 2:
        raise ValueError("Need at least 2 nodes.")


    rng = random.Random(seed)
    repeat_records = []
    all_pair_records = []

    for rep in range(1, n_repeats + 1):
        degree_score_total = 0.0
        bet_score_total = 0.0

        for pair_id in range(1, n_random_pairs + 1):
            node_a, node_b = rng.sample(nodes, 2)

            row_a = df.loc[df["Node"] == node_a].iloc[0]
            row_b = df.loc[df["Node"] == node_b].iloc[0]

            risk_a = row_a["AverageAttackRate"]
            risk_b = row_b["AverageAttackRate"]

            deg_a = row_a["degree_centrality"]
            deg_b = row_b["degree_centrality"]

            bet_a = row_a["betweenness_centrality"]
            bet_b = row_b["betweenness_centrality"]

            actual = "A" if risk_a > risk_b else "B" if risk_b > risk_a else "Tie"
            pred_deg = "A" if deg_a > deg_b else "B" if deg_b > deg_a else "Tie"
            pred_bet = "A" if bet_a > bet_b else "B" if bet_b > bet_a else "Tie"

            degree_score = 1.0 if pred_deg == actual else 0.5 if ("Tie" in [pred_deg, actual]) else 0.0
            bet_score = 1.0 if pred_bet == actual else 0.5 if ("Tie" in [pred_bet, actual]) else 0.0

            degree_score_total += degree_score
            bet_score_total += bet_score

            all_pair_records.append({
                "network": label,
                "repeat_id": rep,
                "pair_id": pair_id,
                "node_A": node_a,
                "node_B": node_b,
                "risk_A": risk_a,
                "risk_B": risk_b,
                "actual_higher_risk": actual,
                "degree_centrality_A": deg_a,
                "degree_centrality_B": deg_b,
                "degree_prediction": pred_deg,
                "degree_score": degree_score,
                "betweenness_centrality_A": bet_a,
                "betweenness_centrality_B": bet_b,
                "betweenness_prediction": pred_bet,
                "betweenness_score": bet_score
            })

        degree_acc = degree_score_total / n_random_pairs
        bet_acc = bet_score_total / n_random_pairs

        repeat_records.append({
            "network": label,
            "repeat_id": rep,
            "n_random_pairs": n_random_pairs,
            "degree_accuracy": degree_acc,
            "betweenness_accuracy": bet_acc,
            "degree_minus_betweenness": degree_acc - bet_acc,
            "degree_better": degree_acc > bet_acc
        })

    repeat_df = pd.DataFrame(repeat_records)
    pair_df = pd.DataFrame(all_pair_records)


    summary_df = pd.DataFrame({
        "network": [label],
        "n_repeats": [n_repeats],
        "n_random_pairs_per_repeat": [n_random_pairs],
        "mean_degree_accuracy": [repeat_df["degree_accuracy"].mean()],
        "std_degree_accuracy": [repeat_df["degree_accuracy"].std(ddof=1)],
        "mean_betweenness_accuracy": [repeat_df["betweenness_accuracy"].mean()],
        "std_betweenness_accuracy": [repeat_df["betweenness_accuracy"].std(ddof=1)],
        "mean_degree_minus_betweenness": [repeat_df["degree_minus_betweenness"].mean()],
        "degree_better_fraction": [repeat_df["degree_better"].mean()],
        "pearson_degree_vs_attack": [df["degree_centrality"].corr(df["AverageAttackRate"], method="pearson")],
        "pearson_betweenness_vs_attack": [df["betweenness_centrality"].corr(df["AverageAttackRate"], method="pearson")],
        "spearman_degree_vs_attack": [df["degree_centrality"].corr(df["AverageAttackRate"], method="spearman")],
        "spearman_betweenness_vs_attack": [df["betweenness_centrality"].corr(df["AverageAttackRate"], method="spearman")]
    })


    merged_path = os.path.join(outdir, f"attack_with_centrality_{label}.csv")
    repeat_path = os.path.join(outdir, f"repeated_pairwise_validation_{label}.csv")
    pair_path = os.path.join(outdir, f"all_pairwise_validation_{label}.csv")
    summary_path = os.path.join(outdir, f"repeated_validation_summary_{label}.csv")

    df.to_csv(merged_path, index=False, float_format="%.6f")
    repeat_df.to_csv(repeat_path, index=False, float_format="%.6f")
    pair_df.to_csv(pair_path, index=False, float_format="%.6f")
    summary_df.to_csv(summary_path, index=False, float_format="%.6f")

    print(f"\n{label} finished.")
    print(summary_df)

    return df, repeat_df, pair_df, summary_df

In [8]:
outdir = "/content/drive/My Drive/Individual_Attack_Rate"

df_sf, repeat_sf, pair_sf, summary_sf = repeated_pairwise_centrality_validation(
    graph_path=os.path.join(outdir, "graph_SF.pkl"),
    attack_path=os.path.join(outdir, "average_attack_SF.csv"),
    label="SF",
    outdir=outdir,
    n_random_pairs=100,
    n_repeats=100,
    seed=42
)

df_sw, repeat_sw, pair_sw, summary_sw = repeated_pairwise_centrality_validation(
    graph_path=os.path.join(outdir, "graph_SW.pkl"),
    attack_path=os.path.join(outdir, "average_attack_SW.csv"),
    label="SW",
    outdir=outdir,
    n_random_pairs=100,
    n_repeats=100,
    seed=42
)


SF finished.
  network  n_repeats  n_random_pairs_per_repeat  mean_degree_accuracy  \
0      SF        100                        100               0.82695   

   std_degree_accuracy  mean_betweenness_accuracy  std_betweenness_accuracy  \
0             0.034479                     0.8319                  0.039992   

   mean_degree_minus_betweenness  degree_better_fraction  \
0                       -0.00495                    0.43   

   pearson_degree_vs_attack  pearson_betweenness_vs_attack  \
0                   0.85683                       0.663451   

   spearman_degree_vs_attack  spearman_betweenness_vs_attack  
0                   0.841859                        0.838295  

SW finished.
  network  n_repeats  n_random_pairs_per_repeat  mean_degree_accuracy  \
0      SW        100                        100                0.7706   

   std_degree_accuracy  mean_betweenness_accuracy  std_betweenness_accuracy  \
0             0.031239                    0.67455                  0

In [9]:
from scipy.stats import ttest_rel

# SF test
t_sf, p_sf = ttest_rel(
    repeat_sf["degree_accuracy"],
    repeat_sf["betweenness_accuracy"]
)

print("SF paired t-test")
print("t-statistic:", t_sf)
print("p-value:", p_sf)


# SW test
t_sw, p_sw = ttest_rel(
    repeat_sw["degree_accuracy"],
    repeat_sw["betweenness_accuracy"]
)

print("\nSW paired t-test")
print("t-statistic:", t_sw)
print("p-value:", p_sw)

SF paired t-test
t-statistic: -1.575910837171161
p-value: 0.11823665315457267

SW paired t-test
t-statistic: 19.72626101060803
p-value: 4.495707192284376e-36
